In [1]:
# https://github.com/d-li14/efficientnetv2.pytorch/blob/main/effnetv2.py
"""
Creates a EfficientNetV2 Model as defined in:
Mingxing Tan, Quoc V. Le. (2021).
EfficientNetV2: Smaller Models and Faster Training
arXiv preprint arXiv:2104.00298.
import from https://github.com/d-li14/mobilenetv2.pytorch
"""

import torch
import torch.nn as nn
import math
from models.linear_layers import MultiToxOutputHead

__all__ = ['efficientnetv2_xs', 'efficientnetv2_s', 'efficientnetv2_m', 'efficientnetv2_l', 'efficientnetv2_xl']


def _make_divisible(v, divisor, min_value=None):
    """
    This function is taken from the original tf repo.
    It ensures that all layers have a channel number that is divisible by 8
    It can be seen here:
    https://github.com/tensorflow/models/blob/master/research/slim/nets/mobilenet/mobilenet.py
    :param v:
    :param divisor:
    :param min_value:
    :return:
    """
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    # Make sure that round down does not go down by more than 10%.
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


# SiLU (Swish) activation function
if hasattr(nn, 'SiLU'):
    SiLU = nn.SiLU
else:
    # For compatibility with old PyTorch versions
    class SiLU(nn.Module):
        def forward(self, x):
            return x * torch.sigmoid(x)


class SELayer(nn.Module):
    def __init__(self, inp, oup, reduction=4):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(oup, _make_divisible(inp // reduction, 8)),
            SiLU(),
            nn.Linear(_make_divisible(inp // reduction, 8), oup),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y


def conv_3x3_bn(inp, oup, stride):
    return nn.Sequential(
        nn.Conv3d(inp, oup, 3, stride, 1, bias=False),
        nn.BatchNorm3d(oup),
        SiLU()
    )


def conv_1x1_bn(inp, oup):
    return nn.Sequential(
        nn.Conv3d(inp, oup, 1, 1, 0, bias=False),
        nn.BatchNorm3d(oup),
        SiLU()
    )


class MBConv(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio, use_se):
        super(MBConv, self).__init__()
        assert stride in [1, 2]

        hidden_dim = round(inp * expand_ratio)
        self.identity = stride == 1 and inp == oup
        if use_se:
            self.conv = nn.Sequential(
                # pw
                nn.Conv3d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm3d(hidden_dim),
                SiLU(),
                # dw
                nn.Conv3d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
                nn.BatchNorm3d(hidden_dim),
                SiLU(),
                SELayer(inp, hidden_dim),
                # pw-linear
                nn.Conv3d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm3d(oup),
            )
        else:
            self.conv = nn.Sequential(
                # fused
                nn.Conv3d(inp, hidden_dim, 3, stride, 1, bias=False),
                nn.BatchNorm3d(hidden_dim),
                SiLU(),
                # pw-linear
                nn.Conv3d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm3d(oup),
            )

    def forward(self, x):
        if self.identity:
            return x + self.conv(x)
        else:
            return self.conv(x)


class EffNetV2(nn.Module):
    """
    t: expand ratio in (Fused-)MBConv block
    c: output channels
    n: number of layers
    s: stride
    use_se: use Fused-MBConv block if True, else use MBConv block

    """
    def __init__(self, effnet_params, config, n_features,
                 channels=3, width_mult=1.):
        super(EffNetV2, self).__init__()
        self.effnet_params = effnet_params
        self.n_features = n_features
        self.use_bias = config.use_bias
        self.lrelu_alpha = config.lrelu_alpha

        # building first layer
        input_channel = _make_divisible(24 * width_mult, 8)
        layers = [conv_3x3_bn(channels, input_channel, 2)]
        # building inverted residual blocks
        block = MBConv
        for t, c, n, s, use_se in self.effnet_params:
            output_channel = _make_divisible(c * width_mult, 8)
            for i in range(n):
                layers.append(block(input_channel, output_channel, s if i == 0 else 1, t, use_se))
                input_channel = output_channel
        self.features = nn.Sequential(*layers)
        # building last several layers
        # output_channel = _make_divisible(1792 * width_mult, 8) if width_mult > 1.0 else 1792
        output_channel = _make_divisible(1280 * width_mult, 8) if width_mult > 1.0 else 1280
        self.conv = conv_1x1_bn(input_channel, output_channel)
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        
        self._initialize_weights()


        # Initialize linear layers
        self.output_head = MultiToxOutputHead(config=config, n_features=n_features)
        


    def forward(self, x, features):
        x = self.forward_backbone(x, features)

        x_dict = self.forward_linear_layers(x, features)

        return x_dict

    def forward_backbone(self, x, features):
        x = self.features(x)
        x = self.conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)

        return x
    
    def forward_linear_layers(self, x, features):
        x_dict = self.output_head(x, features)
        return x_dict


    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                receptive_field_size = 1
                for s in m.kernel_size:
                    receptive_field_size *= s

                n = m.out_channels * receptive_field_size
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.weight.data.normal_(0, 0.001)
                m.bias.data.zero_()


def efficientnetv2_xs(**kwargs):
    """
    Constructs a EfficientNetV2-XS model
    """
    effnet_params = [
        # t, c, n, s, SE
        [1, 12, 2, 1, 0],
        [4, 24, 4, 2, 0],
        [4, 32, 4, 2, 0],
        [4, 64, 6, 2, 1],
        [6, 80, 9, 1, 1],
        [6, 128, 15, 2, 1],
    ]
    return EffNetV2(effnet_params, **kwargs)


def efficientnetv2_s(**kwargs):
    """
    Constructs a EfficientNetV2-S model
    """
    effnet_params = [
        # t, c, n, s, SE
        [1, 24, 2, 1, 0],
        [4, 48, 4, 2, 0],
        [4, 64, 4, 2, 0],
        [4, 128, 6, 2, 1],
        [6, 160, 9, 1, 1],
        [6, 256, 15, 2, 1],
    ]
    return EffNetV2(effnet_params, **kwargs)


def efficientnetv2_m(**kwargs):
    """
    Constructs a EfficientNetV2-M model
    """
    effnet_params = [
        # t, c, n, s, SE
        [1, 24, 3, 1, 0],
        [4, 48, 5, 2, 0],
        [4, 80, 5, 2, 0],
        [4, 160, 7, 2, 1],
        [6, 176, 14, 1, 1],
        [6, 304, 18, 2, 1],
        [6, 512, 5, 1, 1],
    ]
    return EffNetV2(effnet_params, **kwargs)


def efficientnetv2_l(**kwargs):
    """
    Constructs a EfficientNetV2-L model
    """
    effnet_params = [
        # t, c, n, s, SE
        [1, 32, 4, 1, 0],
        [4, 64, 7, 2, 0],
        [4, 96, 7, 2, 0],
        [4, 192, 10, 2, 1],
        [6, 224, 19, 1, 1],
        [6, 384, 25, 2, 1],
        [6, 640, 7, 1, 1],
    ]
    return EffNetV2(effnet_params, **kwargs)


def efficientnetv2_xl(**kwargs):
    """
    Constructs a EfficientNetV2-XL model
    """
    effnet_params = [
        # t, c, n, s, SE
        [1, 32, 4, 1, 0],
        [4, 64, 8, 2, 0],
        [4, 96, 8, 2, 0],
        [4, 192, 16, 2, 1],
        [6, 256, 24, 1, 1],
        [6, 512, 32, 2, 1],
        [6, 640, 8, 1, 1],
    ]
    return EffNetV2(effnet_params, **kwargs)


def get_efficientnetv2(config, model_name, channels, n_features):

    if model_name == 'efficientnetv2_xs':
        model = efficientnetv2_xs(config=config, channels=channels, n_features=n_features)
    elif model_name == 'efficientnetv2_s':
        model = efficientnetv2_s(config=config, channels=channels, n_features=n_features)
    elif model_name == 'efficientnetv2_m':
        model = efficientnetv2_m(config=config, channels=channels, n_features=n_features)
    elif model_name == 'efficientnetv2_l':
        model = efficientnetv2_l(config=config, channels=channels, n_features=n_features)
    elif model_name == 'efficientnetv2_xl':
        model = efficientnetv2_xl(config=config, channels=channels, n_features=n_features)
    else:
        raise ValueError('Model_name = {} is not valid!'.format(model_name))

    return model


In [10]:
import config
from data_preproc.data_preproc_functions import Logger
logger=Logger()

n_features = 10
channels = 3
depth, height, width = 96, 96, 96


model = get_efficientnetv2(config, model_name="efficientnetv2_xl", channels=3, n_features=10)

model.to(device=config.device)
print()



In [11]:
from misc import get_model_summary


total_params = get_model_summary(config=config, model=model, input_size=[(config.batch_size, channels, depth, height, width), (config.batch_size, max(n_features, 1))], 
                                                device=config.device, logger=logger, save_to_file=False)

Layer (type (var_name))                                 Input Shape               Output Shape              Param #                   Kernel Shape
EffNetV2 (EffNetV2)                                     [8, 3, 96, 96, 96]        [8, 1]                    --                        --
├─Sequential (features)                                 [8, 3, 96, 96, 96]        [8, 640, 3, 3, 3]         --                        --
│    └─Sequential (0)                                   [8, 3, 96, 96, 96]        [8, 24, 48, 48, 48]       --                        --
│    │    └─Conv3d (0)                                  [8, 3, 96, 96, 96]        [8, 24, 48, 48, 48]       1,944                     [3, 3, 3]
│    │    └─BatchNorm3d (1)                             [8, 24, 48, 48, 48]       [8, 24, 48, 48, 48]       48                        --
│    │    └─SiLU (2)                                    [8, 24, 48, 48, 48]       [8, 24, 48, 48, 48]       --                        --
│    └─MBConv (1)       

In [5]:
total_params

6104189